# 03 · False alarms on normal Juice Shop traffic

Our `/analyze` test flagged normal Juice Shop requests as attacks. Before the full generalisation test (week 4), we measure how bad this is with **normal traffic only**: every request in the log is normal, so every `attack` is a false alarm.

We compare the two candidate detectors, both trained on CSIC only:
- **both**: XGBoost on TF-IDF + `features.py` (best on CSIC)
- **hand**: XGBoost on `features.py` only (not tied to the CSIC site)

**Input:** `data/raw/juiceshop_normal.log`, an access log from our lab after ~10 minutes of normal browsing. If it is missing, the notebook falls back to `data/raw/sample_access.log`.

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ml.detector import build_pipeline, to_frame
from backend.app.log_parser import parse_log

LOG = ROOT / 'data' / 'raw' / 'juiceshop_normal.log'
if not LOG.exists():
    LOG = ROOT / 'data' / 'raw' / 'sample_access.log'
LOG

## 1. Train both candidates on CSIC

Takes a minute or two. Close Docker Desktop first if memory is tight.

In [ ]:
csic = pd.read_csv(ROOT / 'data' / 'processed' / 'csic_clean.csv', keep_default_na=False)
X = to_frame(csic['Method'], csic['path'], csic['body'])
y = csic['label']
HAND_COLS = [c for c in X.columns if c != 'text']
pos_weight = (y == 0).sum() / (y == 1).sum()

models = {}
for name, use_tfidf in [('both', True), ('hand', False)]:
    t = time.time()
    models[name] = build_pipeline(HAND_COLS, pos_weight=pos_weight, use_tfidf=use_tfidf).fit(X, y)
    print(f'{name}: trained in {time.time() - t:.0f}s')

## 2. Read the Juice Shop log

Juice Shop is a single-page app, so the browser also loads many static files (`.js`, `.css`, images). We report those separately from the API calls (`/rest`, `/api`), which carry the user's actual input.

In [ ]:
rows, skipped = parse_log(LOG.read_text(encoding='utf-8', errors='replace'))
log = pd.DataFrame(rows)
STATIC = ('.js', '.css', '.png', '.jpg', '.jpeg', '.svg', '.ico', '.woff', '.woff2', '.ttf', '.map', '.gif')
def kind(path):
    p = path.split('?')[0].lower()
    if p.startswith(('/rest', '/api')):
        return 'api'
    return 'static' if p.endswith(STATIC) else 'page/other'
log['kind'] = log['path'].map(kind)
print(f'{len(log)} requests parsed, {skipped} lines skipped')
log['kind'].value_counts()

## 3. False-alarm rate

Share of these normal requests that each model calls `attack` (lower is better).

In [ ]:
Xj = to_frame(log['method'], log['path'], log['body'])
for name, m in models.items():
    log[f'p_{name}'] = m.predict_proba(Xj)[:, 1]
    log[f'alarm_{name}'] = log[f'p_{name}'] >= 0.5

rates = pd.DataFrame({name: log.groupby('kind')[f'alarm_{name}'].mean() for name in models})
rates.loc['ALL'] = [log[f'alarm_{n}'].mean() for n in models]
(rates * 100).round(1).rename(columns=lambda c: f'{c} (% false alarms)')

In [ ]:
ax = (rates * 100).plot.bar(rot=0, figsize=(7, 4))
ax.set_ylabel('% of normal requests flagged')
ax.set_title('False alarms on normal Juice Shop traffic')
plt.tight_layout()

## 4. Which normal requests get flagged?

The most confident false alarms for each model. Look for a pattern (a path, a parameter, the body).

In [ ]:
import json

def body_keys(b):
    # show only the field names, never the values (bodies can hold passwords)
    try:
        d = json.loads(b)
        return ','.join(d) if isinstance(d, dict) else type(d).__name__
    except ValueError:
        return f'<{len(b)} chars>' if b else ''

log['body_fields'] = log['body'].map(body_keys)
for name in models:
    print(f'===== {name} =====')
    top = log[log[f'alarm_{name}']].sort_values(f'p_{name}', ascending=False)
    print(top[['method', 'path', 'body_fields', f'p_{name}']].head(10).to_string(max_colwidth=60))
    print()

## 5. Notes

- Which model has fewer false alarms, and on which kind of request?
- If `hand` is much lower, that supports using it (or adding Juice Shop normal traffic to training).
- Next: the full generalisation test with labelled attacks (week 4).